In [ ]:
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.cluster import KMeans
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GroupKFold

from analysis.src.data import complete_case_mask
from analysis.src.io import experiment_paths, load_data
from analysis.src.constants import COVARIATES, OUTCOMES, RANDOM_STATE

In [ ]:
OUT = experiment_paths('prediction/spatial_cross_validation')
DISEASES = OUTCOMES
N_SPATIAL_BLOCKS = 25
N_SPATIAL_FOLDS = 5

data = load_data()

In [ ]:
# Compare out-of-area predictions within shared spatial folds
projected = data.to_crs('EPSG:27700')
centroids = projected.geometry.centroid

incremental_blocks = pd.Series(
    KMeans(n_clusters = N_SPATIAL_BLOCKS, random_state = RANDOM_STATE, n_init = 20).fit_predict(
        np.c_[centroids.x, centroids.y]
    ),
    index = data.index,
)


incremental_blocks.rename('block').to_csv(OUT.spatial_blocks_csv, index_label = 'LSOA21CD')

incremental_rows = []
prediction_rows = []
fold_membership = []
for outcome in DISEASES:
    model_data = data.loc[complete_case_mask(data, outcome, COVARIATES)]

    for fold, (train_i, test_i) in enumerate(
        GroupKFold(n_splits = N_SPATIAL_FOLDS).split(model_data, groups = incremental_blocks.loc[model_data.index]), 1
    ):
        train, test = model_data.iloc[train_i], model_data.iloc[test_i]
        fold_membership.extend({'outcome': outcome, 'fold': fold, 'LSOA21CD': area} for area in test.index)

        predictions = {}

        for label, features in [
            ('Covariates only', COVARIATES),
            ('Covariates + walkability', COVARIATES + ['walkability_score'])
        ]:
            predictions[label] = LinearRegression().fit(train[features], train[outcome]).predict(test[features])

            prediction_rows.append(
                pd.DataFrame(
                    {
                        'outcome': outcome,
                        'fold': fold,
                        'model': label,
                        'LSOA21CD': test.index,
                        'observed': test[outcome].to_numpy(),
                        'predicted': predictions[label]
                    }
                )
            )

            incremental_rows.append(
                {
                    'outcome': outcome,
                    'fold': fold,
                    'model': label,
                    'r_squared': r2_score(test[outcome], predictions[label]),
                    'rmse': mean_squared_error(test[outcome], predictions[label]) ** 0.5,
                    'mae': mean_absolute_error(test[outcome], predictions[label])
                }
            )

incremental_results = pd.DataFrame(incremental_rows)
wide = incremental_results.pivot(index = ['outcome', 'fold'], columns = 'model', values = ['r_squared', 'rmse', 'mae'])

incremental_summary = pd.DataFrame(
    {
        'outcome': wide.index.get_level_values('outcome'),
        'fold': wide.index.get_level_values('fold'),
        'incremental_r_squared': wide['r_squared']['Covariates + walkability'] - wide['r_squared']['Covariates only'],
        'change_rmse': wide['rmse']['Covariates + walkability'] - wide['rmse']['Covariates only'],
        'change_mae': wide['mae']['Covariates + walkability'] - wide['mae']['Covariates only']
    }
).reset_index(drop = True)

incremental_results.merge(incremental_summary, on = ['outcome', 'fold']).to_csv(
    OUT.spatial_cross_validation_csv, index = False
)

display(incremental_summary.groupby('outcome').agg(['mean', 'std']).round(4))

In [ ]:
pd.concat(prediction_rows).to_parquet(OUT.predictions_parquet, index = False)
pd.DataFrame(fold_membership).to_csv(OUT.fold_membership_csv, index = False)